In [ ]:
# This manually installs flams[automl] library as it is not included in the default Colab environment.
!pip install "flaml[automl]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.7/337.7 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: xgboost
    Found existing installation: xgboost 3.2.0
    Uninstalling xgboost-3.2.0:
      Successfully uninstalled xgboost-3.2.0


In [ ]:
# This cell imports necessary dataset and libraries. Do not change it.
# Data source: Quinlan, R. (1993). Auto MPG [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5859H.
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from flaml import AutoML
from sklearn.svm import SVR
from scipy import stats

# fetch dataset
auto_mpg = fetch_ucirepo(id=9)

# data (as pandas dataframes)
X = auto_mpg.data.features
y = auto_mpg.data.targets

In [ ]:
# Data cleaning
# This dataset is already split into X and y. Only X has NaN.
# Drop NaN from both X and corresponding y
df = pd.concat([X, y], axis=1) # This makes the combined dataframe along the columns (axis=1).
df_clean = df.dropna()  # Drop those rows.
X_clean = df_clean.drop(columns=['mpg'])  # Drop the target column.
y_clean = df_clean['mpg']  # Keep only the target column.

In [ ]:
# Split the data into training and test sets.
x_train, x_test, y_train, y_test = train_test_split(X_clean, y_clean, test_size=0.3, random_state=42)

In [ ]:
# #3 Select models
model_aml = AutoML(seed=42,
               time_budget=30, # in seconds
               metric="mae",
               task="regression",
               )
model_svr = SVR()

In [ ]:
# #4 Train models on training set
model_aml.fit(x_train, y_train)
model_svr.fit(x_train, y_train)

[flaml.automl.logger: 03-22 21:55:48] {2375} INFO - task = regression
[flaml.automl.logger: 03-22 21:55:48] {2386} INFO - Evaluation method: cv
[flaml.automl.logger: 03-22 21:55:48] {2489} INFO - Minimizing error metric: mae
[flaml.automl.logger: 03-22 21:55:48] {2606} INFO - List of ML learners in AutoML Run: ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth', 'sgd']
[flaml.automl.logger: 03-22 21:55:48] {2911} INFO - iteration 0, current learner lgbm
[flaml.automl.logger: 03-22 21:55:48] {3046} INFO - Estimated sufficient time budget=6282s. Estimated necessary time budget=45s.
[flaml.automl.logger: 03-22 21:55:48] {3097} INFO -  at 0.7s,	estimator lgbm's best error=4.9569e+00,	best estimator lgbm's best error=4.9569e+00
[flaml.automl.logger: 03-22 21:55:48] {2911} INFO - iteration 1, current learner lgbm
[flaml.automl.logger: 03-22 21:55:49] {3097} INFO -  at 0.9s,	estimator lgbm's best error=4.9569e+00,	best estimator lgbm's best error=4.9569e+00
[flaml.automl.logger: 03-22 21

INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune


[flaml.automl.logger: 03-22 21:55:51] {3097} INFO -  at 2.9s,	estimator sgd's best error=6.7279e+00,	best estimator lgbm's best error=2.5124e+00
[flaml.automl.logger: 03-22 21:55:51] {2911} INFO - iteration 7, current learner lgbm
[flaml.automl.logger: 03-22 21:55:52] {3097} INFO -  at 3.9s,	estimator lgbm's best error=2.3189e+00,	best estimator lgbm's best error=2.3189e+00
[flaml.automl.logger: 03-22 21:55:52] {2911} INFO - iteration 8, current learner sgd
[flaml.automl.logger: 03-22 21:55:52] {3097} INFO -  at 4.4s,	estimator sgd's best error=6.7279e+00,	best estimator lgbm's best error=2.3189e+00
[flaml.automl.logger: 03-22 21:55:52] {2911} INFO - iteration 9, current learner sgd
[flaml.automl.logger: 03-22 21:55:52] {3097} INFO -  at 4.5s,	estimator sgd's best error=5.5938e+00,	best estimator lgbm's best error=2.3189e+00
[flaml.automl.logger: 03-22 21:55:52] {2911} INFO - iteration 10, current learner xgboost
[flaml.automl.logger: 03-22 21:55:53] {3097} INFO -  at 5.7s,	estimator x

SVR()

In [ ]:
# #5 Evaluate models on test set
y_predict_aml = model_aml.predict(x_test)
y_predict_svr = model_svr.predict(x_test)

In [ ]:
# #6 Compute paired absolute errors for each test sample
abs_errors_aml = abs(y_test - y_predict_aml)
abs_errors_svr = abs(y_test - y_predict_svr)

In [ ]:
# #7 Calculate t-statistics and p-value
t_statistics, p_value = stats.ttest_rel(abs_errors_aml, abs_errors_svr, alternative='less')

In [ ]:
print("Paired t-test — Band Gap Prediction Errors: ML vs DL")
print(f"t-statistic = {t_statistics:.4f}")
print(f"p-value = {p_value:.6f}")

Paired t-test — Band Gap Prediction Errors: ML vs DL
t-statistic = -4.6323
p-value = 0.000005


## **Assignment #2: Define and state the null and alternative hypotheses, report the p-value, and decide whether to reject or fail to reject the null hypothesis.**

Answer:

Null hypothesis: There is no significant improvement in predicting car mpg when using AutoML over Support Vector Machine; there is no significant difference in mean absolute error.

Alternative hypothesis: There is significant improvement in predicting car mpg when using AutoML over Support Vector Machine; there is a significantly lower mean absolute error from the AutoML model.

p-value: 0.000010

Since p<0.05, there is sufficient evidence to reject the null hypothesis.